
# OAuth-Protected MCP Servers: Secure Authorization Flow

In the previous notebook (2-mcp-servers.ipynb), we connected to an **unauthenticated** MCP server. But what happens when an MCP server requires OAuth 2.1 authentication?

This is where the **MCP Authorization Specification** comes in - it defines how MCP clients discover OAuth requirements and complete the authorization flow.

In this notebook, we'll walk through the complete OAuth 2.1 authorization flow step by step:

## The 6-Step Authorization Flow

1. **Initial Handshake** - Try to connect without credentials → Get 401 Unauthorized
2. **Protected Resource Metadata (PRM) Discovery** - Learn OAuth requirements
3. **Authorization Server Discovery** - Discover Keycloak endpoints  
4. **Client Registration** - Use pre-registered credentials
5. **User Authorization** - Complete OAuth flow (browser login + token exchange)
6. **Authenticated Requests** - Use access token to call MCP tools

```
┌─────────────┐
│ MCP Client  │ (1) GET /sse (no token)
└──────┬──────┘      ↓
       ↓        401 Unauthorized + PRM URL
┌──────────────────┐  ↑
│   MCP Server     │
│ (OAuth-protected)│
└──────────────────┘
       ↑
       │ (2) Fetch PRM
       │ (3) Discover Keycloak
       │
┌──────┴──────┐
│  Keycloak   │ (5) User logs in
│   (OAuth    │ → Returns access_token
│   Server)   │
└──────┬──────┘
       │
       │ (6) GET /sse
       │     Authorization: Bearer <token>
       ↓
┌──────────────────┐
│   MCP Server     │ → Validates token
│                  │ → Allows request
└──────────────────┘
```

Let's experience each step hands-on!



## Step 0: Import Required Libraries & Configuration

Let's define the URLs we'll be working with:

In [ ]:
import httpx
import json
import asyncio
import base64
import hashlib
import secrets
from urllib.parse import urlencode, parse_qs, urlparse
from datetime import datetime

# Suppress verbose logging
import logging
logging.getLogger("httpx").setLevel(logging.WARNING)

# MCP Server (OAuth-protected)
MCP_SERVER_URL = "http://localhost:3000"
MCP_SSE_ENDPOINT = f"{MCP_SERVER_URL}/sse"

# Keycloak (Authorization Server)
KEYCLOAK_URL = "http://localhost:8080"
KEYCLOAK_REALM = "master"

# Pre-registered OAuth Client
CLIENT_ID = "mcp-client"
CLIENT_SECRET = "zrqR06T0NBTct1ZP08SK8NAc8JfVRW0n"

# Test User Credentials
USERNAME = "testuser"
PASSWORD = "testpassword"

print("📋 Configuration:")
print(f"   MCP Server: {MCP_SERVER_URL}")
print(f"   OAuth Server: {KEYCLOAK_URL}")
print(f"   Client ID: {CLIENT_ID}")
print(f"   Test User: {USERNAME}")

## Step 1: Initial Handshake - "You Need Authorization"

Let's try to connect to the MCP server **without** providing credentials.

What will happen?
- The server will reject us with **401 Unauthorized**
- It will include a `WWW-Authenticate` header telling us where to learn about OAuth requirements

This is the first step in the OAuth discovery flow!

In [ ]:
print("="*80)
print("STEP 1: Initial Handshake (No Credentials)")
print("="*80)

print(f"\n🔌 Attempting to connect to: {MCP_SSE_ENDPOINT}")
print("   (No Authorization header)\n")

# Try to connect without credentials
async with httpx.AsyncClient() as client:
    response = await client.get(MCP_SSE_ENDPOINT)

print(f"📥 Response:")
print(f"   Status Code: {response.status_code}")
print(f"   Status: {response.reason_phrase}\n")

if response.status_code == 401:
    print("✅ Got 401 Unauthorized - Expected!\n")
    
    # Check for WWW-Authenticate header
    www_auth = response.headers.get("WWW-Authenticate")
    if www_auth:
        print("📋 WWW-Authenticate Header:")
        print(f"   {www_auth}\n")
        
        # Extract the PRM URL using string split
        if "resource_metadata=" in www_auth:
            prm_url = www_auth.split('resource_metadata="')[1].split('"')[0]
            print(f"🎯 Protected Resource Metadata (PRM) URL:")
            print(f"   {prm_url}")
    else:
        print("⚠️  No WWW-Authenticate header found")
        prm_url = None
else:
    print(f"⚠️  Unexpected status code: {response.status_code}")
    prm_url = None

## Step 2: PRM Discovery

Now that we know the server requires OAuth, let's fetch the **Protected Resource Metadata (PRM)** document.

This document (defined by RFC 9728) tells us:
- **What** resource this is (the MCP server URL)
- **Where** to get authorization (Keycloak URL)
- **What scopes** are required (`mcp:tools`)
- **How** to send tokens (`Authorization: Bearer` header)

This enables **automatic discovery** - no manual configuration needed!

In [ ]:
print("="*80)
print("STEP 2: Protected Resource Metadata (PRM) Discovery")
print("="*80)

async def fetch_prm_document():
    """Fetch the Protected Resource Metadata document (RFC 9728)."""
    
    # The PRM endpoint follows a standard pattern
    prm_endpoint = f"{MCP_SERVER_URL}/.well-known/oauth-protected-resource"
    
    print(f"\n📥 Fetching PRM document from:")
    print(f"   {prm_endpoint}\n")
    
    async with httpx.AsyncClient() as client:
        response = await client.get(prm_endpoint)
        
        if response.status_code == 200:
            prm_data = response.json()
            
            print("✅ PRM Document Retrieved!\n")
            print("📋 Protected Resource Metadata:")
            print(json.dumps(prm_data, indent=2))
            print()
            
            print("🔍 Specifics:")
            print(f"   • Resource: {prm_data.get('resource')}")
            print(f"   • Authorization Server: {prm_data.get('authorization_servers', [])[0]}")
            print(f"   • Required Scope: {prm_data.get('scopes_supported', [])[0]}")
            print(f"   • Token Method: {prm_data.get('bearer_methods_supported', [])[0]}")
            
            return prm_data
        else:
            print(f"❌ Failed to fetch PRM: {response.status_code}")
            print(f"Response: {response.text}")
            return None

# Fetch the PRM document
prm_data = await fetch_prm_document()

We now know WHERE to get authorization (Keycloak) and WHAT scope we need!

**Next step**: Discover what Keycloak can do.

## Step 3: Authorization Server Discovery

Now that we know Keycloak is our authorization server, let's discover its capabilities.

We'll fetch the **OpenID Connect (OIDC) Discovery** document, which tells us:
- **Authorization endpoint** - Where users log in
- **Token endpoint** - Where we exchange codes for tokens
- **Introspection endpoint** - Where the MCP server validates tokens
- **Supported grant types** - authorization_code (with PKCE)

This is standard OAuth 2.0 / OpenID Connect discovery!

In [ ]:
print("="*80)
print("STEP 3: Authorization Server Discovery (Keycloak)")
print("="*80)

async def discover_authorization_server():
    """Discover Keycloak's OAuth/OIDC endpoints."""
    
    # Standard OIDC discovery endpoint
    discovery_url = f"{KEYCLOAK_URL}/realms/{KEYCLOAK_REALM}/.well-known/openid-configuration"
    
    print(f"\n📥 Fetching OIDC Discovery document from:")
    print(f"   {discovery_url}\n")
    
    async with httpx.AsyncClient() as client:
        response = await client.get(discovery_url)
        
        if response.status_code == 200:
            oidc_config = response.json()
            
            print("✅ OIDC Discovery Successful!\n")
            
            # Extract key endpoints
            print("🔑 Key Endpoints Discovered:")
            print(f"   • Issuer: {oidc_config.get('issuer')}")
            print(f"   • Authorization: {oidc_config.get('authorization_endpoint')}")
            print(f"   • Token: {oidc_config.get('token_endpoint')}")
            print(f"   • Introspection: {oidc_config.get('introspection_endpoint')}")
            print(f"   • Registration: {oidc_config.get('registration_endpoint')}")
            print()
            
            print("📋 Supported Features:")
            print(f"   • Grant Types: {', '.join(oidc_config.get('grant_types_supported', []))}")
            print(f"   • Response Types: {', '.join(oidc_config.get('response_types_supported', []))}")
            print(f"   • PKCE Methods: {', '.join(oidc_config.get('code_challenge_methods_supported', []))}")
            
            return oidc_config
        else:
            print(f"❌ Failed to fetch OIDC config: {response.status_code}")
            return None

# Discover Keycloak endpoints
oidc_config = await discover_authorization_server()

## Step 4: Client Registration

In this step, we need to identify ourselves to Keycloak.

There are two approaches:
1. **Pre-registration** - Client credentials configured ahead of time (what we're using)
2. **Dynamic Client Registration (DCR)** - Runtime registration (more advanced)

In our example (don't do this in production), we're using **pre-registered credentials**:
- Client ID: `mcp-client`
- Client Secret: `zrqR06T0NBTct1ZP08SK8NAc8JfVRW0n`

These were configured in Keycloak ahead of time, so we're ready to go!

Let's verify they work by testing a simple token request.

In [ ]:
print("="*80)
print("STEP 4: Client Registration (Pre-registered)")
print("="*80)

print("\n🆔 Using Pre-registered Client Credentials:")
print(f"   Client ID: {CLIENT_ID}")
print(f"   Client Secret: {CLIENT_SECRET[:20]}...")
print(f"   (Pre-configured in Keycloak)\n")

**Note**: In a production MCP client, you might use:
- Pre-registration (what we're using)
- Dynamic Client Registration (DCR) if supported
- Manual configuration if neither is available

## Step 5: User Authorization

Normally, Step 5 involves:
1. Opening a **browser** for user login
2. User authenticates with Keycloak
3. User grants consent (if needed)
4. Keycloak redirects back with an **authorization code**
5. We exchange the code for an **access token** (with PKCE proof)

This is the **Authorization Code + PKCE flow** - the most secure OAuth flow.

However, for simplicity in this notebook, we'll use the **Password Grant** flow, which:
- Skips the browser redirect
- Directly exchanges username/password for a token
- Is simpler for demos but **not recommended for production**

In [ ]:
print("="*80)
print("STEP 5: User Authorization (Password Grant - Simplified)")
print("="*80)

async def get_access_token_password_grant():
    """Get an access token using Password Grant (for demo purposes)."""
    
    token_endpoint = oidc_config.get('token_endpoint')
    
    print(f"\n🔐 Requesting Access Token via Password Grant")
    print(f"   Endpoint: {token_endpoint}")
    print(f"   Username: {USERNAME}")
    print(f"   Scope: mcp:tools\n")
    
    async with httpx.AsyncClient() as client:
        response = await client.post(
            token_endpoint,
            data={
                "grant_type": "password",
                "client_id": CLIENT_ID,
                "client_secret": CLIENT_SECRET,
                "username": USERNAME,
                "password": PASSWORD,
                "scope": "mcp:tools",
            },
            headers={"Content-Type": "application/x-www-form-urlencoded"},
        )
        
        if response.status_code == 200:
            token_data = response.json()
            
            print("✅ Access Token Obtained!\n")
            print("📊 Token Information:")
            print(f"   Access Token: {token_data['access_token'][:60]}...")
            print(f"   Token Type: {token_data['token_type']}")
            print(f"   Expires In: {token_data['expires_in']} seconds")
            print(f"   Scope: {token_data.get('scope', 'N/A')}\n")
            
            return token_data
        else:
            print(f"❌ Failed to get token: {response.status_code}")
            print(f"Response: {response.text}")
            return None

# Get the access token
token_data = await get_access_token_password_grant()
access_token = token_data['access_token'] if token_data else None

We successfully obtained an access token!

This token proves we're authorized and can now call the MCP server.

**Next step**: Use the token to make authenticated requests.

## Token Introspection: How the MCP Server Validates Tokens

Before we use our token, let's see how the **MCP server validates it**.

When we send a request with our access token, the MCP server:
1. Extracts the token from the `Authorization: Bearer` header
2. Calls Keycloak's **introspection endpoint** (RFC 7662)
3. Validates:
   - Token is **active** (not expired/revoked)
   - **Audience** matches the MCP server URL
   - **Scope** includes required permissions (`mcp:tools`)
4. Returns the validated token metadata

Let's introspect our token to see what the MCP server sees!

In [ ]:
print("="*80)
print("Token Introspection: What the MCP Server Sees")
print("="*80)

async def introspect_token(access_token):
    """Introspect the access token (what the MCP server does)."""
    
    introspection_endpoint = oidc_config.get('introspection_endpoint')
    
    print(f"\n🔍 Introspecting Token at Keycloak")
    print(f"   Endpoint: {introspection_endpoint}\n")
    
    async with httpx.AsyncClient() as client:
        response = await client.post(
            introspection_endpoint,
            data={
                "token": access_token,
                "client_id": CLIENT_ID,
                "client_secret": CLIENT_SECRET,
            },
            headers={"Content-Type": "application/x-www-form-urlencoded"},
        )
        
        if response.status_code == 200:
            introspection_data = response.json()
            
            print("✅ Token Introspection Successful!\n")
            print("📋 Token Metadata (What MCP Server Validates):")
            print(json.dumps(introspection_data, indent=2))
            print()
            
            print("🔐 Validation Checks (MCP Server Logic):")
            print(f"   ✅ Active: {introspection_data.get('active')}")
            print(f"   ✅ Client ID: {introspection_data.get('client_id')}")
            print(f"   ✅ Username: {introspection_data.get('username')}")
            print(f"   ✅ Scope: {introspection_data.get('scope')}")
            print(f"   ✅ Audience: {introspection_data.get('aud')}")
            print(f"   ✅ Expires At: {datetime.fromtimestamp(introspection_data.get('exp', 0))}")
            
            return introspection_data
        else:
            print(f"❌ Introspection failed: {response.status_code}")
            print(f"Response: {response.text}")
            return None

# Introspect the token
if access_token:
    introspection_data = await introspect_token(access_token)
    # print("\n📋 Access Token:")
    # print(access_token)

This is what the MCP server sees when validating our token!

If all checks pass, our request will be allowed through.

## Step 6: Using the Authenticated MCP Client

Now that we have a valid token, we can use the **MCP Python SDK** to interact with the server!

The SDK handles:
- SSE connection management
- Message serialization
- Tool discovery
- Tool invocation

All we need to do is provide our access token in the headers.

In [ ]:
print("="*80)
print("Using Authenticated MCP Client")
print("="*80)

from mcp import ClientSession
from mcp.client.sse import sse_client

async def list_mcp_tools_authenticated(access_token):
    """List MCP tools using authenticated connection."""
    
    print(f"\n🔐 Connecting to MCP Server with OAuth Token...\n")
    
    # Create headers with our access token
    headers = {
        "Authorization": f"Bearer {access_token}"
    }
    
    try:
        # Connect to MCP server with authentication
        async with sse_client(
            MCP_SSE_ENDPOINT,
            headers=headers  # 👈 Pass our OAuth token!
        ) as (read, write):
            async with ClientSession(read, write) as session:
                # Initialize the session
                await session.initialize()
                print("✅ Authenticated MCP Session Established!\n")
                
                # List available tools
                tools = await session.list_tools()
                
                print(f"📋 Found {len(tools.tools)} OAuth-Protected MCP Tools:\n")
                print("-" * 80)
                
                for i, tool in enumerate(tools.tools, 1):
                    print(f"{i}. {tool.name}")
                    print(f"   {tool.description}")
                    print("-" * 80)
                
                return session, tools
    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        import traceback
        traceback.print_exc()
        return None, None

# List tools with authentication
if access_token:
    session, tools = await list_mcp_tools_authenticated(access_token)
    
    print("\n" + "="*80)
    print("📌 Success!")
    print("   We can now discover and call OAuth-protected MCP tools!")
    print("="*80)
else:
    print("\n⚠️  No access token available - cannot connect to MCP server")

## Step 7: Calling MCP Tools (With Authentication)

Now let's actually call an MCP tool with our authenticated connection!

We'll call `get_calendar_statistics` to see calendar stats.

In [ ]:
print("="*80)
print("Calling MCP Tools (Authenticated)")
print("="*80)

from datetime import datetime, timedelta

async def call_mcp_tool_authenticated(access_token, tool_name, arguments=None):
    """Call an MCP tool with authentication."""
    
    print(f"\n📞 Calling MCP Tool: {tool_name}")
    if arguments:
        print(f"   Arguments: {json.dumps(arguments, indent=2)}")
    print()
    
    headers = {
        "Authorization": f"Bearer {access_token}"
    }
    
    try:
        async with sse_client(MCP_SSE_ENDPOINT, headers=headers) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                
                # Call the tool
                result = await session.call_tool(
                    tool_name,
                    arguments=arguments or {}
                )
                
                print("📊 Result:")
                print("-" * 80)
                for content in result.content:
                    print(content.text)
                print("-" * 80)
                
                return result
    except Exception as e:
        print(f"❌ Error calling tool: {e}")
        import traceback
        traceback.print_exc()
        return None

# Example 1: Get calendar statistics (read operation)
if access_token:
    print("\n🎯 Example 1: Getting Calendar Statistics\n")
    
    result = await call_mcp_tool_authenticated(
        access_token,
        "get_calendar_statistics"
    )
    
    print("\n" + "="*80)
    print("✅ Successfully Called OAuth-Protected MCP Tool (Read)!")
    print("="*80)


In [ ]:
# Example 2: Create a new event (write operation)
print("\n" + "="*80)
print("🎯 Example 2: Creating a New Event")
print("="*80)

from datetime import datetime, timedelta

if access_token:
    # Prepare event data (same format as 2-mcp-servers.ipynb)
    event_name = "OAuth Authentication Workshop"
    tomorrow = datetime.now() + timedelta(days=1)
    start_time = tomorrow.replace(hour=10, minute=0, second=0)
    end_time = start_time + timedelta(hours=2)
    
    event_data = {
        "name": event_name,
        "content": "Learn about OAuth 2.1 authentication with MCP servers",
        "category": "Workshop",
        "level": 3,  # High priority
        "start_time": start_time.strftime("%Y-%m-%d %H:%M:%S"),
        "end_time": end_time.strftime("%Y-%m-%d %H:%M:%S")
    }
    
    print(f"\nCreating event: {event_name}")
    print(f"Time: {event_data['start_time']} → {event_data['end_time']}\n")
    
    # Create the event
    result = await call_mcp_tool_authenticated(
        access_token,
        "create_event",
        arguments=event_data
    )
    
    print("\n" + "="*80)
    print("✅ Successfully Created Event with OAuth Authentication!")
    print("="*80)
